# StableHLO Export

## What We Can Inspect: Reading the StableHLO

In [1]:
import jax
import jax.numpy as jnp


def f(x, w, b):
    return jnp.tanh(x @ w + b)

In [ ]:
print(jax.make_jaxpr(f)(jnp.ones((4, 8)), jnp.ones((8, 4)), jnp.ones(4)))

{ lambda ; a:f32[4,8] b:f32[8,4] c:f32[4]. let
    d:f32[4,4] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a b
    e:f32[1,4] = broadcast_in_dim[broadcast_dimensions=(1,)] c
    f:f32[4,4] = add d e
    g:f32[4,4] = tanh f
  in (g,) }


In [ ]:
lowered = jax.jit(f).lower(jnp.ones((4, 8)), jnp.ones((8, 4)), jnp.ones(4))

print(lowered.as_text())

module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<4x8xf32>, %arg1: tensor<8x4xf32>, %arg2: tensor<4xf32>) -> (tensor<4x4xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [1] x [0], precision = [DEFAULT, DEFAULT] : (tensor<4x8xf32>, tensor<8x4xf32>) -> tensor<4x4xf32>
    %1 = stablehlo.broadcast_in_dim %arg2, dims = [1] : (tensor<4xf32>) -> tensor<1x4xf32>
    %2 = stablehlo.broadcast_in_dim %1, dims = [0, 1] : (tensor<1x4xf32>) -> tensor<4x4xf32>
    %3 = stablehlo.add %0, %2 : tensor<4x4xf32>
    %4 = stablehlo.tanh %3 : tensor<4x4xf32>
    return %4 : tensor<4x4xf32>
  }
}



## After the Optimizer: Reading the Compiled Program

In [4]:
compiled = lowered.compile()
print(compiled.as_text())

HloModule jit_f, is_scheduled=true, entry_computation_layout={(f32[4,8]{1,0}, f32[8,4]{1,0}, f32[4]{0})->f32[4,4]{1,0}}, allow_spmd_sharding_propagation_to_parameters={true,true,true}, allow_spmd_sharding_propagation_to_output={true}

FileNames
1 "<frozen runpy>"
2 "C:\\Users\\user\\AppData\\Local\\Temp\\ipykernel_8320\\2735958025.py"
3 "C:\\Users\\user\\AppData\\Local\\Temp\\ipykernel_8320\\1459353492.py"

FunctionNames
1 "_run_module_as_main"
2 "_run_code"
3 "<module>"
4 "f"

FileLocations
1 {file_name_id=1 function_name_id=1 line=198 end_line=199 column=11 end_column=42}
2 {file_name_id=1 function_name_id=2 line=88 end_line=88 column=4 end_column=27}
3 {file_name_id=2 function_name_id=3 line=1 end_line=1 column=10 end_column=73}
4 {file_name_id=3 function_name_id=4 line=6 end_line=6 column=20 end_column=25}
5 {file_name_id=3 function_name_id=4 line=6 end_line=6 column=11 end_column=30}

StackFrames
1 {file_location_id=1 parent_frame_id=1}
2 {file_location_id=2 parent_frame_id=2}
3 {

## Asking the Compiler What It Costs

In [ ]:
n = 256


@jax.jit
def matmul(A, B):
    return A @ B


compiled = jax.jit(matmul).lower(jnp.ones((n, n)), jnp.ones((n, n))).compile()

In [6]:
cost = compiled.cost_analysis()
print(f"flops reported: {cost["flops"]}")
print(f"2*n^3: {2*n ** 3}")
print(f"bytes accessed: {cost["bytes accessed"]}")

flops reported: 33554432.0
2*n^3: 33554432
bytes accessed: 786432.0


In [7]:
@jax.jit
def chain(x):
    x = jnp.sin(x)
    x = jnp.exp(x)
    x = x * 2.0
    return jnp.tanh(x)

In [ ]:
x = jnp.ones((1000, 1000))  # 4 MB of float32

cost = jax.jit(chain).lower(x).compile().cost_analysis()
print(f"bytes accessed: {cost["bytes accessed"] / 1e6:1f} MB")

bytes accessed: 8.000000 MB


In [ ]:
mem = compiled.memory_analysis()
print(f"arguments: {mem.argument_size_in_bytes}")
print(f"output: {mem.output_size_in_bytes}")
print(f"temporaries: {mem.temp_size_in_bytes}")

# dir(mem)

arguments: 524288
output: 262144
temporaries: 0


## Model Export with jax.export

In [ ]:
from jax.export import export


@jax.jit
def model(x, params):
    return jnp.tanh(x @ params["w"] + params["b"])

In [ ]:
x_shape = jax.ShapeDtypeStruct((4, 8), jnp.float32)
params_shape = {
    "w": jax.ShapeDtypeStruct((8, 4), jnp.float32),
    "b": jax.ShapeDtypeStruct((4,), jnp.float32),
}

In [12]:
exported = export(model)(x_shape, params_shape)

In [13]:
# dir(exported)

In [14]:
type(exported)

jax._src.export._export.Exported

## Inspecting and Calling an Exported Module

In [15]:
print("Input shapes:")
for aval in exported.in_avals:
    print(f" {aval}")

Input shapes:
 float32[4,8]
 float32[4]
 float32[8,4]


In [16]:
print("Output shapes:")
for aval in exported.out_avals:
    print(f" {aval}")

Output shapes:
 float32[4,4]


In [ ]:
import jax.random as jrandom

key = jrandom.PRNGKey(0)
x = jrandom.normal(key, (4, 8))
params = {"w": jrandom.normal(key, (8, 4)), "b": jnp.zeros(4)}

In [18]:
result = exported.call(x, params)
print(f"result: {result}")

result: [[ 0.99954206 -0.34840393 -0.9995229   0.27433315]
 [ 0.5852951  -0.36324698  0.9373974  -0.99999833]
 [-0.9998186   0.98156774 -0.99960905  0.98729825]
 [ 0.9038114  -0.99818474  0.7446091  -0.8613436 ]]


In [19]:
reference = model(x, params)
print(f"reference result: {reference}")
print()
print(f"Max diff: {jnp.max(jnp.abs(result - reference))}")

reference result: [[ 0.99954206 -0.34840393 -0.9995229   0.27433315]
 [ 0.5852951  -0.36324698  0.9373974  -0.99999833]
 [-0.9998186   0.98156774 -0.99960905  0.98729825]
 [ 0.9038114  -0.99818474  0.7446091  -0.8613436 ]]

Max diff: 0.0


## Dynamic Shapes in Export

In [24]:
(batch,) = jax.export.symbolic_shape("batch")

In [25]:
batch

batch

In [ ]:
exported_dynamic = export(model)(
    jax.ShapeDtypeStruct((batch, 8), jnp.float32),
    {
        "w": jax.ShapeDtypeStruct((8, 4), jnp.float32),
        "b": jax.ShapeDtypeStruct((4,), jnp.float32),
    },
)

In [28]:
print("Input shapes:")
for aval in exported_dynamic.in_avals:
    print(f" {aval}")

Input shapes:
 float32[batch,8]
 float32[4]
 float32[8,4]


In [29]:
print("Output shapes:")
for aval in exported_dynamic.out_avals:
    print(f" {aval}")

Output shapes:
 float32[batch,4]


## When StableHLO Matters in Practice

In [ ]:
@jax.jit
def production_fn(x):
    return jnp.sqrt(jnp.sum(x**2, axis=-1))

In [ ]:
# Case 1: Checking that a function lowers correctly before deployment
lowered = jax.jit(production_fn).lower(jnp.ones((100, 64)))
# there was a mistake in the recording (typo: "succeded" -> "succeeded")
print("Lowering succeeded: StableHLO is valid")

Lowering succeeded: StableHLO is valid


In [39]:
# Case 2: Checking for unexpected operations
print(f"StableHLO: {lowered.as_text()}")

StableHLO: module @jit_production_fn attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<100x64xf32>) -> (tensor<100xf32> {jax.result_info = "result"}) {
    %0 = call @production_fn(%arg0) : (tensor<100x64xf32>) -> tensor<100xf32>
    return %0 : tensor<100xf32>
  }
  func.func private @production_fn(%arg0: tensor<100x64xf32>) -> tensor<100xf32> {
    %0 = stablehlo.multiply %arg0, %arg0 : tensor<100x64xf32>
    %cst = stablehlo.constant dense<0.000000e+00> : tensor<f32>
    %1 = stablehlo.reduce(%0 init: %cst) applies stablehlo.add across dimensions = [1] : (tensor<100x64xf32>, tensor<f32>) -> tensor<100xf32>
    %2 = stablehlo.sqrt %1 : tensor<100xf32>
    return %2 : tensor<100xf32>
  }
}



In [41]:
# Case 3: Serializing and deserializing an exported model
exported = export(jax.jit(production_fn))(jax.ShapeDtypeStruct((100, 64), jnp.float32))

x = jnp.ones((100, 64))
reference = production_fn(x)
print(f"reference result: {reference}")
print()

serialized = exported.serialize()
print(f"Serialized bytes: {len(serialized)}")

del exported, production_fn

reloaded = jax.export.deserialize(serialized)
result = reloaded.call(x)
print(f"Results match: {jnp.allclose(reference, result)}")

reference result: [8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8. 8.
 8. 8. 8. 8.]

Serialized bytes: 1608
Results match: True
